### Import Libraries

In [0]:
from pyspark.sql.functions import md5, concat, concat_ws, col

### Function to check if file exits or not

In [0]:
def filesCheck(path):
    if path != None:
        try:
            dbutils.fs.ls(path)
            #display(dbutils.fs.ls(path))
            return True
        except:
            return False
    else:
        return False

In [0]:
path_user = 'dbfs:/user/'
filesCheck(path_user)

In [0]:
%fs ls

In [0]:
def DeltaCheck(path):    
    """This function takes path as argument and check if delta table exits at this path or not"""
    return DeltaTable.isDeltaTable(spark, path)

In [0]:
data = [('James','','Smith','1991-04-01','M',3000),
  ('Michael','Rose','','2000-05-19','M',4000),
  ('Robert','','Williams','1978-09-05','M',4000),
  ('Maria','Anne','Jones','1967-12-01','F',4000),
  ('Jen','Mary','Brown','1980-02-17','F',-1)]

columns = ["firstname","middlename","lastname","dob","gender","salary"]
df = spark.createDataFrame(data = data, schema = columns)
df.printSchema()
df.show()

### concat() and concat_ws()
1. concat() - used to concat dataframes mutiple columns into a single column
2. concat_ws() - concat with seperator, used to concat dataframes with mutiple columns with a seperator

In [0]:
df_concat =  df.select(concat(col("firstname"), col("middlename"), col("lastname")).alias('Full Name'))
df_concat.show()

In [0]:
df_concat_ws =  df.select(concat_ws('-',df.firstname, df.middlename, df.lastname).alias("Full Name with -"))
df_concat_ws.show()

In [0]:
df_concat_ws =  df.select(concat_ws('-',(col("firstname"), col("middlename"), col("lastname")))).alias("Full Name with -")
df_concat_ws.show()

In [0]:
# to get the list of all the columns of dataframe 
[c for c in df.columns]

In [0]:
# to get the list of all the columns of dataframe using pyspark 
[col(c) for c in df.columns]

In [0]:
df_concat_all =  df.select(concat(col("firstname"), col("middlename"), col("lastname")).alias('Full Name'))
df_concat_all.show()

In [0]:
df.select(concat([(col(c)).cast("string") for c in df.columns]).display()

In [0]:
df_md5 = df.withColumn("md5", concat_ws('~',*[md5((col(c)).cast("string")) for c in df.columns]))

In [0]:
df_md5.display()

### generate md5 function

In [0]:
def generateMd5(df):
    """This function first converts all columns into string along with calculating md5 values and then concatenates them using concat_ws using '~' symbol.
    After this it adds new column as md5 to the dataframe and returns both count and new dataframe """
    try:
        df = df.select(sorted(df.columns, reverse=False))
        df = df.withColumn("md5", concat_ws('~',*[md5((col(c)).cast("string")) for c in df.columns]))
        return df.count(), df
    except Exception as e:
        raise e

In [0]:
count, df_md5 = generateMd5(df)
print(count)
df_md5.display()

In [0]:
count, df_md5 = generateMd5(df)
print(count)
df_md5.display()

### Data Sources

Various types of sources of ingestion are 
1. Batch data ingestion
2. Streaming data ingestion - Real-time data ingestion using Spark structured streaming which can process data from live sources 
3. Cloud storage ingestion
4. Relational databases ingestion
5. NoSQL databases ingestion
6. Cloud data warehouses ingestion

#### JDBC data ingestion 

JDBC is a java API that provides classes and functions for java programs to interact with relational databases.

In [0]:
username = 'rohan_test'
password = 'Test@123'
driver = {'sql_server':'com.microsoft.sqlserver.jdbc.SQLServerDriver',
          'mysql':'',
          '':'',

host_name = '10.70.11.192'
port = 1433
database_name = 'test_database' 
instance_name = 'TEST'

url = f"jdbc:sqlserver://{host_name}:{port};database={database_name};instance_name={instance_name}"

connection_properties = {
    "user" : username,
    "password" : password,
    "driver" : driver
}
print(url)
print(connection_properties)

In [0]:
def readSqlServer(username, password, driver, url, query):
    tableSrc = spark.read.format("jdbc")\
                    .option("driver", driver)\
                    .option("url", url)\
                    .option("query",query)\
                    .option("user", username)\
                    .option("password", password)\
                    .option("trustServerCertificate", "true")\
                    .load()
    return tableSrc, tableSrc.count()

In [0]:
def readSqlServerSpark(query, driver, url, username, password,):
    tableSrc = spark.read.format("com.microsoft.sqlserver.jdbc.spark")\
                    .option("driver", driver)\
                    .option("url", url)\
                    .option("query",query)\
                    .option("user", username)\
                    .option("password", password)\
                    .option("trustServerCertificate", "true")\
                    .load()
    return tableSrc, tableSrc.count()

In [0]:
def readSqlServerConnectionString(connection_string, table_name):
    table = spark.read.jdbc(connection_string, table_name)
    return table

### archiving function


create a table with c

In [0]:
%fs ls '/user/metastore'

In [0]:
dbutils.fs.mkdirs('/user/metastore/config')

In [0]:
dbuti